# 4.1 章节介绍

## 概述

通过第3章学习，已经掌握了 Vector 算子的编写，并完成向量分核、分块、流水排布。本章把同样的方法迁移到矩阵乘：先跑通基础 GEMM 的实现，再实现核内多输出块的流水，最后进行 Persistent 调度优化。

## 学习目标

- 了解 Cube 计算单元输入、计算和输出的内存位置。
- 循序渐进掌握基础 GEMM 实现、流水排布，以及 Persistent 的背景和方法。

前置：会 Python、已学习第3章，能读懂局部工作区与分块搬运；不要求已有 Cube 算子开发经验。有卡学员重跑代码，无卡学员阅读真实输出与答案，实操能力仍需之后上机确认。

[本章导航](README.md) · [下一节](04.02_basic_gemm.ipynb)

<p><small>执行前完成第1.4节环境准备，按本章README的统一入口运行。每页使用新内核；NPU程序由独立子进程执行，notebook内核不初始化设备。</small></p>

<img alt="同一个 GEMM 的三步演进" src="images/04.01_route_v3.svg" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

基础版确定 tiling 分核，流水版尝试进行流水排布，实现搬运和计算的掩盖，Persistent 调度版改变逻辑块遍历任务的方式。

## 1. 本章路线

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">小节</th>
<th style="text-align: left; vertical-align: top;">学习内容</th>
<th style="text-align: left; vertical-align: top;">实践与验证</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="04.02_basic_gemm.ipynb">4.2</a></td>
<td style="text-align: left; vertical-align: top;">基础 GEMM：tiling 分核、输入搬运与分块累加</td>
<td style="text-align: left; vertical-align: top;">推导输出块和 K 循环的索引，运行基础 GEMM 并验证完整输出</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="04.03_pipeline.ipynb">4.3</a></td>
<td style="text-align: left; vertical-align: top;">流水排布：核内多个输出块的搬运与计算并行</td>
<td style="text-align: left; vertical-align: top;">保持相同矩阵乘问题，对比流水配置，检查结果与 kernel 耗时</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="04.04_persistent.ipynb">4.4</a></td>
<td style="text-align: left; vertical-align: top;">Persistent：背景与输出任务的遍历方法</td>
<td style="text-align: left; vertical-align: top;">推导任务编号，实现 Persistent 并验证任务覆盖与结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="04.05_practice.ipynb">4.5</a></td>
<td style="text-align: left; vertical-align: top;">GEMM 综合实践：实现与调优</td>
<td style="text-align: left; vertical-align: top;">扩展到 36 核，再调整 BM，联合左矩阵复用与列优先任务分配，比较耗时、带宽和算力利用率</td>
</tr>
</tbody>
</table>

## 2. 本章的矩阵乘用例

本章围绕 C=AB<sup>T</sup> 展开：A 的每一行与 B 的每一行做点积，得到 C 中对应位置的元素。设 A 有 M 行、B 有 N 行，两者每行均有 K 项，则 C 有 M 行、N 列。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">矩阵</th>
<th style="text-align: left; vertical-align: top;">实际存储形状</th>
<th style="text-align: left; vertical-align: top;">数据类型与作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">A</td>
<td style="text-align: left; vertical-align: top;"><code>(M,K)</code></td>
<td style="text-align: left; vertical-align: top;">bf16 输入</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">B</td>
<td style="text-align: left; vertical-align: top;"><code>(N,K)</code></td>
<td style="text-align: left; vertical-align: top;">bf16 输入，计算时使用逻辑转置 B<sup>T</sup></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">C</td>
<td style="text-align: left; vertical-align: top;"><code>(M,N)</code></td>
<td style="text-align: left; vertical-align: top;">fp32 输出，计算过程也采用 fp32 累加</td>
</tr>
</tbody>
</table>

例如，4.2 的基础用例中，A 为 `(512,256)`、B 为 `(36864,256)`，输出 C 为 `(512,36864)`。B 的实际存储形状仍为 `(36864,256)`，逻辑转置不等于事先将存储改成 `(256,36864)`。

基础版、流水版和 Persistent 调度版都计算上述矩阵乘：依次学习如何分核、如何安排核内多输出块的搬运与计算、如何组织逻辑块遍历任务。在对照不同实现时，保持用例输入一致，再验证结果并比较性能。

4.2～4.4 使用 8 个核任务讲解原理；4.5 将核数扩展到 Ascend 950DT_9592 实际的 36 个 Cube 核。输出 tile 数取所用核数的倍数，以便每核承担相同数量的完整任务。输入为连续、有限的 bf16 矩阵，输出为 fp32。

### 输入范围与边界

**本章不处理元素尾块**：M、N 分别被 BM、BN 整除；采用 K 切分时，K 被 BK 整除。这是为集中说明原理而选定的输入范围，不是矩阵乘本身的限制。非整除输入需要边界保护、K 尾块补零和有效区域写回。

基础、流水、Persistent 与章末实践统一使用 `(M,N,K)=(512,36864,256)`；K 切分说明保持 M、N 不变，将 K 改为 2048。每组对照保持相同输入，不跨 shape 互算加速比。

## 小结

本章从 Vector 算子走向 Cube 矩阵乘。先在基础 GEMM 中认识输入块、累加结果与输出块的内存位置，完成 tiling 分核和 K 维分块累加；再实现搬运与计算流水并行，通过 Persistent 调度进行优化；最后扩展到实际 36 核，在同一用例上完成 tiling 与输入复用调优。三个实现版本始终保持 C=AB<sup>T</sup> 的数学含义，先验证输出结果正确性，再评价优化效果。

## 课后练习

1. 本章计算 C=AB<sup>T</sup>。当 A 为 `(640,512)`、B 为 `(1024,512)` 时，C 的形状是什么？B 的实际存储形状与 B<sup>T</sup> 的逻辑形状有什么区别？
2. 对照本章路线，说明基础 GEMM、多 buffer 流水和 Persistent 调度分别要解决什么问题。三种实现都必须保持什么不变？

先结合本章用例与路线回答，再查看解析。

In [1]:
from pathlib import Path
print(Path("answer/04.01_answer.txt").read_text())

1. C 的形状为 (640,1024)；B 实际为 (1024,512)，逻辑转置为 (512,1024)。
2. 基础版确定输出分块和 8 核分工；流水版让多个输出块的搬运与计算交叠；Persistent 改变任务编号的遍历。三者保持同一数学结果。K 切分保留为容量与累加说明。

